# BirdCLEF+ 2026 — 234-class multi-label classifier (V4: ECA-NFNet-L0 + pseudo-labels)

Builds on the V3 setup (v1 mel params, pseudo-labels) using ECA-NFNet-L0 backbone to provide
a stronger diverse member for the V3+V4 ensemble.

Inputs:
- Competition data via `kagglehub`.
- `species-001-010` + `species-011-090`: v1 mel spectrogram PNGs from `train_audio`.
- `soundscape-spectrograms`: labeled soundscape PNGs + `soundscape_index.csv`.
- `pseudo-labels-part1` + `pseudo-labels-part2`: ensemble-labeled unlabeled soundscape clips
  (confidence ≥ 0.5), stored as zip batches + `pseudo_label_index.csv`.

Pseudo-label clips are used for training only — never validation.
Validation uses only the original labeled data (same as V1/V5 baseline).

Outputs: `model_multilabel_234.pkl` and `vocab.json`.

## Setup

In [ ]:
! pip install -Uqq fastbook timm
import fastbook
fastbook.setup_book()

In [ ]:
import ast
import hashlib
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

from fastbook import *
from fastai.vision.all import *
import json
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

## Paths

In [ ]:
import kagglehub
competition_dir = Path(kagglehub.competition_download('birdclef-2026'))
print('Competition dir:', competition_dir)

SPECIES_FOLDERS = [
    Path('/kaggle/input/datasets/ucheozoemena/species-001-010'),
    Path('/kaggle/input/datasets/ucheozoemena/species-011-090'),
]
SOUNDSCAPE_DATASET_DIR = Path('/kaggle/input/datasets/ucheozoemena/soundscape-spectrograms')
PSEUDO_LABEL_DIRS = [
    Path('/kaggle/input/datasets/ucheozoemena/pseudo-labels-part1'),
    Path('/kaggle/input/datasets/ucheozoemena/pseudo-labels-part2'),
]

for p in SPECIES_FOLDERS + [SOUNDSCAPE_DATASET_DIR] + PSEUDO_LABEL_DIRS:
    print(f'{p.name}: exists={p.exists()}')

## Build the unified multi-label dataframe

Three sources:
- **train_audio**: each PNG → one positive label (parent folder name) + any secondary labels from `train.csv`.
- **soundscapes**: each PNG → semicolon-joined multi-label string from `soundscape_index.csv`.
- **pseudo**: each PNG → semicolon-joined labels from ensemble inference (confidence ≥ 0.5). Train split only.

The `group` column drives the validation split via a stable hash so clips from the same recording
always stay together.

In [ ]:
MAX_PER_SPECIES = 500

_valid_species = set(
    c for c in pd.read_csv(competition_dir / 'sample_submission.csv').columns
    if c != 'row_id'
)

def _parse_secondaries(val):
    if pd.isna(val) or str(val).strip() in ('', '[]'):
        return []
    try:
        result = ast.literal_eval(val)
        return [s for s in result if s and s in _valid_species]
    except Exception:
        return []

_train_csv = pd.read_csv(competition_dir / 'train.csv')
_secondary_lookup = dict(zip(
    _train_csv['filename'].apply(lambda f: Path(f).stem),
    _train_csv['secondary_labels'].apply(_parse_secondaries),
))

isolated_rows = []
for folder in SPECIES_FOLDERS:
    if not folder.exists():
        print(f'Skipping missing folder: {folder}')
        continue
    for png in get_image_files(folder):
        primary = png.parent.name
        secondaries = [s for s in _secondary_lookup.get(png.stem, []) if s != primary]
        isolated_rows.append({
            'image_path': str(png),
            'labels': ';'.join([primary] + secondaries),
            'primary_label': primary,
            'source': 'train_audio',
            'group': primary,
        })
isolated_df = pd.DataFrame(isolated_rows)

isolated_df = (
    isolated_df
    .groupby('primary_label', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_SPECIES), random_state=42))
    .reset_index(drop=True)
    .drop(columns='primary_label')
)

print(f'train_audio rows (capped at {MAX_PER_SPECIES}/species): {len(isolated_df)}')
print(f'unique primary species: {isolated_df["labels"].str.split(";").str[0].nunique()}')

In [ ]:
soundscape_index = pd.read_csv(SOUNDSCAPE_DATASET_DIR / 'soundscape_index.csv')

def soundscape_group(image_name: str) -> str:
    parts = image_name.split('_')
    return '_'.join(parts[:3])

soundscape_df = pd.DataFrame({
    'image_path': [str(SOUNDSCAPE_DATASET_DIR / p) for p in soundscape_index['image_path']],
    'labels': soundscape_index['labels'],
    'source': 'soundscape',
    'group': soundscape_index['image_path'].map(soundscape_group),
})
print(f'soundscape rows: {len(soundscape_df)}')

In [ ]:
# Kaggle auto-extracts zips on dataset upload — scan dirs recursively for PNGs
png_lookup = {}
for pl_dir in PSEUDO_LABEL_DIRS:
    for png_path in pl_dir.rglob('*.png'):
        png_lookup[png_path.name] = str(png_path)
print(f'Found {len(png_lookup):,} PNGs across pseudo-label datasets')

index_parts = []
for pl_dir in PSEUDO_LABEL_DIRS:
    idx_path = pl_dir / 'pseudo_label_index.csv'
    if idx_path.exists():
        index_parts.append(pd.read_csv(idx_path))
    else:
        print(f'WARNING: {idx_path} not found')
pseudo_index = pd.concat(index_parts, ignore_index=True)
print(f'Pseudo-label index: {len(pseudo_index):,} rows')

pseudo_index['full_path'] = pseudo_index['image_path'].map(png_lookup)
n_missing = int(pseudo_index['full_path'].isna().sum())
if n_missing > 0:
    print(f'WARNING: {n_missing:,} index entries have no matching PNG — dropping them')
pseudo_index = pseudo_index.dropna(subset=['full_path']).reset_index(drop=True)

def pseudo_group(png_name: str) -> str:
    return png_name.rsplit('__k', 1)[0]

pseudo_df = pd.DataFrame({
    'image_path': pseudo_index['full_path'],
    'labels': pseudo_index['labels'],
    'source': 'pseudo',
    'group': pseudo_index['image_path'].map(pseudo_group),
})
print(f'Pseudo-label rows: {len(pseudo_df):,}')
print(f'Unique soundscape groups: {pseudo_df["group"].nunique():,}')

In [ ]:
VAL_FRACTION = 0.2
SEED = 42

def stable_hash_in_valid(group: str, frac: float) -> bool:
    h = hashlib.md5(f'{group}:{SEED}'.encode()).digest()
    bucket = int.from_bytes(h[:4], 'big') / 2**32
    return bucket < frac

df = pd.concat([isolated_df, soundscape_df, pseudo_df], ignore_index=True)
df['is_valid'] = df['group'].map(lambda g: stable_hash_in_valid(g, VAL_FRACTION))
# Pseudo-labels are always train-only — clean validation uses only labeled data
df.loc[df['source'] == 'pseudo', 'is_valid'] = False

print(f'total rows: {len(df):,} (train={int((~df.is_valid).sum()):,}, valid={int(df.is_valid.sum()):,})')
print('source x split:')
print(df.groupby(['source', 'is_valid']).size())

## Pin vocabulary to the 234 sample-submission classes

In [ ]:
sample_sub = pd.read_csv(competition_dir / 'sample_submission.csv')
ALL_SPECIES = [c for c in sample_sub.columns if c != 'row_id']
assert len(ALL_SPECIES) == 234, f'expected 234, got {len(ALL_SPECIES)}'

labels_seen = set()
for s in df['labels']:
    labels_seen.update(s.split(';'))
unknown = labels_seen - set(ALL_SPECIES)
assert not unknown, f'labels not in sample_submission columns: {unknown}'
missing_in_data = set(ALL_SPECIES) - labels_seen
print(f'classes in vocab but absent from training data: {len(missing_in_data)}')
if missing_in_data:
    print('  example:', sorted(missing_in_data)[:10])

## DataBlock and dataloaders

In [ ]:
BATCH_SIZE = 64
IMG_SIZE = 224

dblock = DataBlock(
    blocks=(ImageBlock, MultiCategoryBlock(vocab=ALL_SPECIES)),
    splitter=ColSplitter('is_valid'),
    get_x=ColReader('image_path'),
    get_y=ColReader('labels', label_delim=';'),
    item_tfms=Resize(IMG_SIZE),
    batch_tfms=aug_transforms(size=IMG_SIZE, do_flip=False, max_rotate=0.0),
)
dls = dblock.dataloaders(df, bs=BATCH_SIZE)

assert list(dls.vocab) == ALL_SPECIES, 'vocab order must match sample_submission column order'
print(f'vocab size: {len(dls.vocab)}  train batches: {len(dls.train)}  valid batches: {len(dls.valid)}')

## Class imbalance via BCE `pos_weight`

`pos_weight[c] = (N - n_c) / n_c`, clipped to `[1, 50]`. Computed only on the training split.

In [ ]:
train_df = df[~df['is_valid']]
N = len(train_df)
pos_counts = pd.Series(0, index=ALL_SPECIES, dtype='int64')
for s in train_df['labels']:
    for lab in s.split(';'):
        pos_counts[lab] += 1

pos_counts = pos_counts.replace(0, 1)
raw = (N - pos_counts) / pos_counts
pos_weight = raw.clip(lower=1.0, upper=50.0).astype('float32')
pos_weight_t = torch.tensor(pos_weight.values, device=dls.device)

print('pos_weight summary:')
print(pos_weight.describe())
print('\nlowest-positive classes (most upweighted):')
print(pos_counts.sort_values().head(15))

## Learner

In [ ]:
loss_func = BCEWithLogitsLossFlat(pos_weight=pos_weight_t)
metrics = [accuracy_multi, APScoreMulti(average='macro')]

learn = vision_learner(
    dls,
    'eca_nfnet_l0',
    loss_func=loss_func,
    metrics=metrics,
    n_out=len(ALL_SPECIES),
).to_fp16()
learn.summary

## Train

In [ ]:
learn.fine_tune(2)

In [ ]:
learn.fit_one_cycle(5, lr_max=1e-4)

## Export model + vocab

In [ ]:
learn.export('model_multilabel_234.pkl')
with open('vocab.json', 'w') as fp:
    json.dump(list(dls.vocab), fp)
print('Saved model_multilabel_234.pkl and vocab.json')

## Verification: per-class AP for the missing-28

The 28 species absent from `train_audio` are the hardest to learn. We compute their AP on the
validation split (original labeled data only) to check whether pseudo-label supervision helped.

In [ ]:
train_csv = pd.read_csv(competition_dir / 'train.csv')
taxonomy_csv = pd.read_csv(competition_dir / 'taxonomy.csv')
train_audio_species = set(train_csv['primary_label'].unique())
missing28 = sorted(set(taxonomy_csv['primary_label']) - train_audio_species)
assert len(missing28) == 28, f'expected 28 missing, got {len(missing28)}'

preds, targs = learn.get_preds(dl=dls.valid)
preds_np = preds.cpu().numpy()
targs_np = targs.cpu().numpy()
vocab_index = {name: i for i, name in enumerate(dls.vocab)}

report_rows = []
for sp in missing28:
    j = vocab_index[sp]
    n_pos = int(targs_np[:, j].sum())
    ap = float(average_precision_score(targs_np[:, j], preds_np[:, j])) if n_pos > 0 else float('nan')
    report_rows.append({'species': sp, 'val_positives': n_pos, 'ap': ap})

report = pd.DataFrame(report_rows).sort_values('ap', na_position='last')
print(report.to_string(index=False))

low_ap = report[(report['val_positives'] > 0) & (report['ap'] < 0.05)]
print(f'\nMissing-28 with AP < 0.05: {len(low_ap)}')

In [ ]:
aps = []
for j in range(targs_np.shape[1]):
    if targs_np[:, j].sum() == 0:
        continue
    aps.append(average_precision_score(targs_np[:, j], preds_np[:, j]))
print(f'Macro-AP over {len(aps)} classes with validation positives: {np.mean(aps):.4f}')